# Physics-Driven Motion on Pretrained Gaussian Splats

This notebook presents two baseline experiments that apply simple physics-driven motion rules to a pretrained Gaussian Splatting representation of the ficus scene.

The goal is not to build a fully physically accurate simulator, but to study how direct motion updates affect the rendered appearance of a recognizable Gaussian Splat object.

## Notebook Scope and Assumptions

This notebook assumes the pretrained ficus scene is available under `output/ficus_whitebg-trained/`, including `cameras.json` and the pretrained Gaussian point cloud.

Rendered PNG frames are written to `output/ficus_whitebg-trained/results/`, and the export cells convert those frame sequences into MP4 videos under `assets/demos/`.

In [3]:
import os
from pathlib import Path

import torchvision
from gaussian_splatting import GaussianModel, Camera
import torch


CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR
if not (REPO_ROOT / "output").exists() and (CURRENT_DIR.parent / "output").exists():
    REPO_ROOT = CURRENT_DIR.parent
os.chdir(REPO_ROOT)

SCENE_ROOT = REPO_ROOT / "output" / "ficus_whitebg-trained"
RESULTS_DIR = SCENE_ROOT / "results"
CAMERAS_PATH = SCENE_ROOT / "cameras.json"
POINT_CLOUD_PATH = SCENE_ROOT / "point_cloud" / "iteration_30000" / "point_cloud.ply"
DEMO_DIR = REPO_ROOT / "assets" / "demos"

def render(gaussians: GaussianModel, camera: Camera, camera_id: int, pyhs_iter: int, save: str):
    # Render the current Gaussian state and save the frame to disk.
    os.makedirs(save, exist_ok=True)
    out = gaussians(camera)
    rendering = out['render']
    torchvision.utils.save_image(rendering, os.path.join(save, f'{camera_id}' + '_{0:05d}'.format(pyhs_iter) + ".png"))

## Experiment 1: Uniform Gravity With Position Clamp

This baseline applies the same gravity vector to every Gaussian and clamps the updated positions to a bounded spatial region.

It serves as a reference case for globally consistent motion and helps show how a simple shared update rule affects the rendered structure of the object.

In [4]:
from gaussian_splatting.dataset import JSONCameraDataset

# Experiment configuration
dt = .02
camera_id = 0
model_results = str(RESULTS_DIR)

# Load the pretrained Gaussian representation and define the gravity direction
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -1., 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        # Apply a constant gravity update to the velocity
        velocity += gravity * dt
        # Keep the motion inside a bounded region
        points = torch.clamp(gaussians.get_xyz + velocity * dt, min=-5., max=5.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 1 Video

The following cell converts the rendered PNG sequence from Experiment 1 into an MP4 file for qualitative inspection.

In [5]:
# Export the rendered frame sequence for Experiment 1
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wall_smash.mp4')} -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## Experiment 2: Randomized Inverse-Mass Motion

This baseline keeps the same rendering pipeline but scales motion by a randomized inverse-mass term for each Gaussian.

Unlike Experiment 1, this produces nonuniform motion across the object and makes local variation more visible in the rendered result.

In [6]:
# Experiment configuration
dt = .02
camera_id = 2
model_results = str(RESULTS_DIR)

# Load the pretrained Gaussian representation and define the gravity direction
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., 0., -1.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]

camera = cameras[camera_id]
inv_mass = 1 / torch.rand((n, 1), device=gaussians.get_xyz.device)

velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        # Apply a constant gravity update to the velocity
        velocity += gravity * dt
        # Scale the position update by randomized inverse mass
        points = torch.clamp(gaussians.get_xyz + velocity * inv_mass * dt, min=-10., max=10.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 2 Video

The following cell converts the rendered PNG sequence from Experiment 2 into an MP4 file for qualitative comparison against the uniform-gravity baseline.

In [7]:
# Export the rendered frame sequence for Experiment 2
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'mass_falling.mp4')} -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## Experiment 3: Wind Field Deformation

This baseline applies a smooth wind field to the Gaussian cloud and uses light velocity damping to avoid unbounded drift.

It serves as a coherent external-force case and helps show how lateral flow changes the rendered structure of the object.

In [16]:
# Experiment configuration
dt = .02
camera_id = 0
model_results = str(RESULTS_DIR / "wind_field")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply a smooth wind field with light damping to keep the motion stable
        gust_x = 0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08)
        gust_z = 0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05)
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 3 Video

The following cell converts the rendered PNG sequence from Experiment 3 into an MP4 file for qualitative inspection of the wind-driven deformation.

In [17]:
# Export the rendered frame sequence for Experiment 3
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field.mp4')} -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

## Experiment 4: Wind Strength Ablation

This experiment reuses the wind-field setup from Experiment 3 and varies only the wind magnitude.

We treat Experiment 3 as the medium-wind baseline and compare it against lower and higher wind strengths to study how force magnitude affects visual coherence.

### Low Wind Variant

This variant reduces the wind magnitude while keeping the rest of the pipeline unchanged.

In [26]:
# Experiment configuration
dt = .02
camera_id = 0
wind_scale = 0.1
model_results = str(RESULTS_DIR / "wind_field_low")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply the Experiment 3 wind field at reduced strength
        gust_x = wind_scale * (0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08))
        gust_z = wind_scale * (0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05))
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Low Wind Video

The following cell converts the rendered PNG sequence from the low-wind variant into an MP4 file.

In [27]:
# Export the rendered frame sequence for the low-wind variant
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field_low.mp4')} -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

### High Wind Variant

This variant increases the wind magnitude while keeping the rest of the pipeline unchanged.

In [28]:
# Experiment configuration
dt = .02
camera_id = 0
wind_scale = 4.0
model_results = str(RESULTS_DIR / "wind_field_high")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply the Experiment 3 wind field at increased strength
        gust_x = wind_scale * (0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08))
        gust_z = wind_scale * (0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05))
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export High Wind Video

The following cell converts the rendered PNG sequence from the high-wind variant into an MP4 file.

In [29]:
# Export the rendered frame sequence for the high-wind variant
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field_high.mp4')} -y

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena